In [1]:
from keras.models import Model 
from keras.layers import Input, Convolution2D, MaxPooling2D, Dense, Dropout, Flatten, Dense, Dropout, Flatten, Conv2D, MaxPooling2D
# import np_utils
import numpy as np
import pandas as pd
from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.optimizers import Adam
from keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler
import os
import datetime
from sklearn.model_selection import train_test_split

from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import SpectralClustering


from sklearn.utils import resample
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import pairwise_distances_argmin_min
from tslearn.metrics import cdist_dtw

In [2]:
poor = pd.read_csv("SimData/bank_reserves_outputs_poor.csv", header=None)
middle = pd.read_csv("SimData/bank_reserves_outputs_middle.csv", header=None)
rich = pd.read_csv("SimData/bank_reserves_outputs_rich.csv", header=None)
sc = StandardScaler()
br = []
for i in np.arange(0, poor.shape[0]):
    sample = pd.concat([poor.iloc[i], middle.iloc[i]], axis=0).T
    sample = pd.concat([sample, rich.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    br.append(sample_std)

In [3]:
ecv_active = pd.read_csv("SimData/epsteinCV_outputs_active.csv", header=None)
ecv_jailed = pd.read_csv("SimData/epsteinCV_outputs_jailed.csv", header=None)
ecv_quiet = pd.read_csv("SimData/epsteinCV_outputs_quiet.csv", header=None)
sc = StandardScaler()
ecv = []
for i in np.arange(0, ecv_active.shape[0]):
    sample = pd.concat([ecv_active.iloc[i], ecv_jailed.iloc[i]], axis=0).T
    sample = pd.concat([sample, ecv_quiet.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    ecv.append(sample_std)

In [4]:
def import_ff_data(filename):
    expected_columns=155
    data = []
    with open(filename, 'r') as file:
        for line in file:
            row = line.strip().split(',')
            if len(row) < expected_columns:
                row += [np.nan] * (expected_columns - len(row))
            data.append(row)
    df = pd.DataFrame(data)
    def fill_last_valid(row):
        for i in range(1, len(row)):
            if pd.isna(row[i]):
                row[i] = row[i - 1]  
        return row
    df_filled = df.apply(fill_last_valid, axis=1)
    return df_filled

In [5]:
ff_onfire = import_ff_data("SimData/forest_fire_outputs_onfire.csv")
print("check 1")
ff_fine = import_ff_data("SimData/forest_fire_outputs_fine.csv")
print("check 2")
ff_burned = import_ff_data("SimData/forest_fire_outputs_burned.csv")
sc = StandardScaler()
ff = []
for i in np.arange(0, ff_onfire.shape[0]):
    sample = pd.concat([ff_onfire.iloc[i], ff_fine.iloc[i]], axis=0).T
    sample = pd.concat([sample, ff_burned.iloc[i]], axis=0).T
    sample_std = sc.fit_transform(sample.to_frame())
    ff.append(sample_std)

check 1
check 2


In [6]:
# Place raw features for all three models in a single place so we can iterate over them 

ABMs = ["BR", "ECV", "FF"]
raw_data = []
raw_data.append(br)
raw_data.append(ecv)
raw_data.append(ff)

In [7]:
RANDOM_STATE = 42

In [8]:
# Extract PCA embeddings; not re-doing PCA in this notebook
pca_br_embeds = pd.read_csv('extracted_features/bank_reserves_pca_standardized.csv')
pca_br_embeds = pca_br_embeds.drop('Unnamed: 0', axis=1)
pca_br_embeds = pca_br_embeds.to_numpy()

pca_ecv_embeds = pd.read_csv('extracted_features/epstein_pca_standardized.csv')
pca_ecv_embeds = pca_ecv_embeds.drop('Unnamed: 0', axis=1)
pca_ecv_embeds = pca_ecv_embeds.to_numpy()

pca_ff_embeds = pd.read_csv('extracted_features/forestfire_pca_standardized.csv')
pca_ff_embeds = pca_ff_embeds.drop('Unnamed: 0', axis=1)
pca_ff_embeds = pca_ff_embeds.to_numpy()

pca_embeds = []
pca_embeds.append(pca_br_embeds)
pca_embeds.append(pca_ecv_embeds)
pca_embeds.append(pca_ff_embeds)

In [10]:
# Extract DAE embeddings; not re-doing DAE in this notebook
dae_br_embeds = pd.read_csv('extracted_features/bank_reserves_DAE.csv')
#dae_br_embeds = dae_br_embeds.drop('Unnamed: 0', axis=1)
dae_br_embeds = dae_br_embeds.to_numpy()

dae_ecv_embeds = pd.read_csv('extracted_features/epstein_DAE.csv')
#dae_ecv_embeds = dae_ecv_embeds.drop('Unnamed: 0', axis=1)
dae_ecv_embeds = dae_ecv_embeds.to_numpy()

dae_ff_embeds = pd.read_csv('extracted_features/forestfire_DAE.csv')
#dae_ff_embeds = dae_ff_embeds.drop('Unnamed: 0', axis=1)
dae_ff_embeds = dae_ff_embeds.to_numpy()

dae_embeds = []
dae_embeds.append(dae_br_embeds)
dae_embeds.append(dae_ecv_embeds)
dae_embeds.append(dae_ff_embeds)

In [11]:
# Extract DAE+ embeddings; not re-doing DAE+ in this notebook
daep_br_embeds = pd.read_csv('extracted_features/0_DAE_5_2_2048_linear.csv', header=None)

daep_ecv_embeds = pd.read_csv('extracted_features/1_DAE_5_3_2048_linear.csv', header=None)

daep_ff_embeds = pd.read_csv('extracted_features/2_DAE_3_3_2048_linear.csv', header=None)

daep_embeds = []
daep_embeds.append(daep_br_embeds)
daep_embeds.append(daep_ecv_embeds)
daep_embeds.append(daep_ff_embeds)

In [ ]:
# Silouhette score - DTW
from tslearn.clustering import silhouette_score
debug = 1
sil_sample = 20000 # sample 

def sil_score_dtw(X_embed, labels, s): 
    SILH_SAMPLE = s 
    if SILH_SAMPLE is not None and SILH_SAMPLE < X_embed.shape[0]:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(X_embed.shape[0], size=SILH_SAMPLE, replace=False)
        sil = silhouette_score(X_embed[idx], labels[idx], metric="soft-dtw")
    else:
        sil = silhouette_score(X_embed, labels, metric="soft-dtw")

    unique, counts = np.unique(labels, return_counts=True)
    cluster_sizes = dict(zip(unique.tolist(), counts.tolist()))
    if debug:
        print(f"Silhouette Score: {sil:.4f}")
      #  print("Cluster sizes:", cluster_sizes)
    return sil

In [ ]:
def sil_score(X_embed, labels, s): 
    SILH_SAMPLE = s 
    if SILH_SAMPLE is not None and SILH_SAMPLE < X_embed.shape[0]:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(X_embed.shape[0], size=SILH_SAMPLE, replace=False)
        sil = silhouette_score(X_embed[idx], labels[idx], metric="euclidean")
    else:
        sil = silhouette_score(X_embed, labels, metric="euclidean")

    unique, counts = np.unique(labels, return_counts=True)
    cluster_sizes = dict(zip(unique.tolist(), counts.tolist()))
    if debug:
        print(f"Silhouette Score: {sil:.4f}")
      #  print("Cluster sizes:", cluster_sizes)
    return sil

In [68]:
# Get silhouette scores for raw features
for i, abm in enumerate(ABMs): 
    for k in range(3,11):
        flat_list = [sample.flatten() for sample in raw_data[i]]
        raw_features = np.asarray(flat_list)
        raw_labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(raw_features).labels_
        unique_labels = np.unique(raw_labels)
        if (len(unique_labels) > 1): 
            score_kmeans_euc = sil_score(pca_embeds[i], raw_labels, sil_sample)
            score_kmeans_dtw = sil_score_dtw(pca_embeds[i], raw_labels, sil_sample)
        else: 
            print(f"Error: Too few labels generated. Skipping ...")
        print(f"[PCA,{abm},{k}] \t Silhouette Score: {score:.4f}")  
        with open("raw_sil_scores.csv", "a") as f:
            f.write(f"{abm},{k},{score:.4f}\n")

MemoryError: Unable to allocate 2.98 GiB for an array with shape (20000, 20000) and data type float64

In [70]:
# Get silhouette scores for pca features 
for i, abm in enumerate(ABMs): 
    for k in range(3,11):
        pca_labels_kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(pca_embeds[i]).labels_
        unique_labels_kmeans = np.unique(pca_labels_kmeans)
        if (len(unique_labels) > 1): 
            score_kmeans_euc = sil_score(pca_embeds[i], pca_labels_kmeans, sil_sample)
            #score_kmeans_dtw = sil_score_dtw(pca_embeds[i], pca_labels_kmeans, sil_sample)
        else: 
            print(f"Error: Too few labels generated. Skipping ...")
        print(f"[PCA,{abm},{k}] \t Silhouette Score: {score:.4f}")
        with open("pca_sil_scores.csv", "a") as f:
            f.write(f"{abm},{k},{score:.4f}\n")

MemoryError: Unable to allocate 2.98 GiB for an array with shape (20000, 20000) and data type float64